In [1]:
using Revise
using InteractiveUtils

const PATH_AUGMENT_QOG_JL = "phase0/functions/qog_augmented_standard.jl"
const PATH_PDF_EXTRACT_JL = "phase0/functions/qog_pdf_extract.jl"
const PATH_METADATA_ENHANCE_JL = "phase0/functions/qog_metadata_join.jl"

includet(PATH_METADATA_ENHANCE_JL)

╔══════════════════════════════════════════════════════════════════════════╗
║ QoG METADATA JOINING - PHASE 0 LOADED                                ║
╚══════════════════════════════════════════════════════════════════════════╝

Quick Start:
    metadata = join_metadata()              # Run full pipeline (single isomorphism check)
    metadata = join_metadata_with_cascade()  # Run cascade (strictest → loosest), then union on slug
    quick_check()                             # Diagnostic check
    inspect_exceptions()                      # Review configuration
    show_usage()                              # Detailed documentation

Pipeline Steps:
    1. ingest_and_normalize()            # Load & normalize sources (PDF = qog_slugs_temporal.csv; min_year/max_year ingested)
    2. align_id_variables!(...)          # Harmonize ID vars
    3. run_isomorphism_cascade(...)      # Strictest → loosest until success; returns (stata_df, pdf_df, arrow_df) for union on slug
    4. unify_and_join(..

In [2]:

# Step 1: Ingest and normalize (lowercase, ligature→ASCII, SLUG_CORRECTIONS; PDF_ONLY_PREFIXES removed; returns 4th: pdf_prefixes_df)
(stata_df, pdf_df, arrow_df, pdf_prefixes_df) = ingest_and_normalize();

PHASE 0: QoG METADATA JOINING - INGESTION

>>> Loading source files...
    Stata manifest:  2009 rows
    PDF extraction:  2119 rows
    Arrow slugs:     2013 rows
    PDF prefixes:    117 rows

>>> Normalizing Stata manifest...
    Normalized: 2009 slugs

>>> Normalizing PDF extraction...
    Removed 44 rows from PDF (prefixes only in PDF: ens, gdg, jht, qs20)
    Normalized: 2075 slugs (after PDF-only prefix removal)
    PDF temporal: 1891 slugs with valid (min_year, max_year)

>>> Normalizing Arrow slugs...
    Normalized: 2013 slugs

>>> Normalizing PDF prefixes file...
    Normalized: 117 prefix rows


In [3]:

# Step 2: Align ID variables
align_id_variables!(stata_df, pdf_df, arrow_df);


>>> Aligning identification variables...
    Stata: Prefixed 9 ID variables with 'ident_' (slug and prefix)
    PDF: Added 9 ID variables from Arrow


In [4]:

# Quick file check (without processing)
quick_check();

# Review exception configurations
inspect_exceptions();

QUICK DIAGNOSTIC CHECK

File sizes:
  Stata:  2009 rows, 3 columns
  PDF:    2119 rows, 7 columns
  Arrow:  2013 rows, 3 columns

Stata columns:  Variable, Label, Prefix
PDF columns:    slug, prefix, description, type, provenance, min_year, max_year
Arrow columns:  slug, prefix, type

EXCEPTION CONFIGURATION REVIEW

EXCLUDED_PREFIXES (1 items):
  - ggis_

PDF_ONLY_PREFIXES (removed from pdf_df on ingest, 4 items):
  - ens
  - gdg
  - jht
  - qs20

SLUG_CORRECTIONS (0 items):
  (none)

DEPRECATED_SLUGS (1 items):
  - who_roadtrd

UNDOCUMENTED_SLUGS (1 items):
  - whr_hap

ADDITIONAL_SLUG_METADATA (1 items):
  - whr_hap

GGIS_METADATA (0 items):
  (none)

ID_VARIABLES (9 items):
  - ident_cname_qog
  - ident_cname
  - ident_year
  - ident_ccodecow
  - ident_ccodealp
  - ident_ccodealp_year
  - ident_ccode_qog
  - ident_cname_year
  - ident_ccode



In [9]:

# Step 3: Validate isomorphism (optional: check_column=:slug or :prefix for single-column check)
# validate_isomorphism(stata_df, pdf_df, arrow_df; check_column=:slug);   # slug-only
validate_isomorphism(stata_df, pdf_df, arrow_df; check_column=:prefix); # prefix-only


PHASE 0: ISOMORPHISM VALIDATION (3-WAY CHECK)
    (check_column = :prefix)

>>> Filtering Arrow to exclude custom prefixes...
    Arrow before filtering: 2013 slugs
    Arrow after filtering:  2009 slugs
    Excluded: 4 slugs

>>> Building prefix sets...
    Stata set:  116 unique prefixes
    PDF set:    115 unique prefixes
    Arrow set:  116 unique prefixes

>>> Applying exception handling...

>>> Computing symmetric differences...

✅ SUCCESS: All sources are isomorphic!
    Aligned: 115 prefixes


In [12]:

# Step 3b: Two-phase isomorphism (full vs trimmed: PDF-only without temporal removed)
# cmp = compare_isomorphism_with_temporal(stata_df, pdf_df, arrow_df);

In [5]:

# Step 3b: Isomorphism cascade (strictest → loosest until success); returns 3 isomorphic dfs for union on slug
result = run_isomorphism_cascade(stata_df, pdf_df, arrow_df)
stata_df, pdf_df, arrow_df = result.stata_df, result.pdf_df, result.arrow_df;


PHASE 0: ISOMORPHISM CASCADE (STRICTEST → LOOSEST)

>>> Trying Phase 1 (full PDF)...

PHASE 0: ISOMORPHISM VALIDATION (3-WAY CHECK)

>>> Filtering Arrow to exclude custom prefixes...
    Arrow before filtering: 2013 slugs
    Arrow after filtering:  2009 slugs
    Excluded: 4 slugs

>>> Building (slug, prefix) sets...
    Stata set:  2009 unique (slug, prefix) pairs
    PDF set:    2084 unique (slug, prefix) pairs
    Arrow set:  2009 unique (slug, prefix) pairs

>>> Applying exception handling...
    Removed 1 deprecated slugs from PDF set
    Removed 1 undocumented slugs from Stata set
    Removed 1 undocumented slugs from Arrow set

>>> Computing symmetric differences...

❌ VALIDATION FAILED: Set mismatches detected

📌 In PDF but NOT in Stata (75):
    - (epi_air, epi)
    - (epi_for, epi)
    - (eu_heaalcday, eu)
    - (eu_heaalcmon, eu)
    - (eu_heaalcnv, eu)
    - (eu_heaalcwk, eu)
    - (eu_heasmok, eu)
    - (pei_eir_2, pei)
    - (sgi_cb24, sgi)
    - (sgi_co24, sgi)
    - (

In [6]:

# Step 4: Unify and enrich (union on slug)
metadata = unify_and_join(stata_df, pdf_df, arrow_df);


PHASE 0: UNIFICATION & JOINING

>>> Merging Stata and PDF on slug...
    Merged: 2010 total rows

>>> Finalizing schema...
    Final schema: ["slug", "prefix", "label", "description", "type", "provenance", "min_year", "max_year"]
    Total variables: 2010


In [7]:

# Step 5: Unify prefixes (metadata-driven; enriches with PDF docs; injects ident/ggis)
prefix_df = unify_prefixes(metadata, pdf_prefixes_df);


PHASE 0: PREFIX UNIFICATION
>>> Found 116 active prefixes (excluding: base, missing)
>>> Injected metadata for 1 non-QoG prefix(es): ident, ggis
>>> Prefix unification complete. Schema: ["prefix", "datasource", "source_name", "citation", "last_update", "description", "provenance"]


In [8]:

# Save manually
CSV.write(PATH_METADATA_JOINED, metadata)
CSV.write(PATH_PREFIX_JOINED, prefix_df)

"data/qog_prefix_joined.csv"

In [9]:
# show_usage()